# Redrob Ranker — Dense Embedding Precompute (GPU)

Produces the 3 dense artifacts consumed by `rank.py`:
`candidate_embeddings.npy`, `candidate_ids.txt`, `jd_embedding.npy` (+ `model_name.txt`).

Model: **BAAI/bge-small-en-v1.5** (384-d). The `evidence_text()` and `JD_QUERY`
below are copied **verbatim** from `src/features.py` and `src/jd_spec.py` so the
artifacts align exactly with what the local ranker expects.

**Steps:** Runtime → Change runtime type → **T4 GPU**. Then run all cells.
Upload `candidates.jsonl.gz` (~52 MB) when prompted. At the end it downloads
`redrob_artifacts.zip` — unzip it into the repo's `artifacts/` folder.

In [ ]:
!pip -q install "sentence-transformers>=2.2"
import torch
print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# Upload candidates.jsonl.gz (preferred) or candidates.jsonl
from google.colab import files
up = files.upload()
CANDIDATES = list(up.keys())[0]
print('using', CANDIDATES)

In [ ]:
import gzip, json

# --- verbatim from src/jd_spec.py ---
JD_QUERY = (
    "Senior AI engineer who has shipped production embeddings-based retrieval, "
    "hybrid and vector search, and ranking, search, recommendation or personalization "
    "systems to real users at a product company. Strong Python engineer who has "
    "designed evaluation frameworks for ranking systems using NDCG, MRR, MAP and "
    "A/B testing, and has handled embedding drift, index refresh and retrieval-quality "
    "regression in production. Around six to eight years of applied machine learning "
    "experience at product companies rather than pure research or services. Based in "
    "or willing to relocate to Pune, Noida, Hyderabad, Mumbai, Delhi NCR or Bangalore."
)

# --- verbatim from src/features.py ---
def evidence_text(c):
    p = c.get("profile", {})
    parts = [p.get("headline", ""), p.get("summary", "")]
    for r in c.get("career_history", []) or []:
        parts.append(f"{r.get('title','')}. {r.get('description','')}")
    return " ".join(parts)

def load(path):
    op = gzip.open if path.endswith('.gz') else open
    with op(path, 'rt', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)

cands = list(load(CANDIDATES))
ids = [c['candidate_id'] for c in cands]
texts = [evidence_text(c) for c in cands]
print(len(ids), 'candidates loaded; example id', ids[0])

In [ ]:
import numpy as np, os, time
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('BAAI/bge-small-en-v1.5', device='cuda')
t0 = time.time()
emb = model.encode(texts, batch_size=256, show_progress_bar=True,
                   normalize_embeddings=True).astype('float32')
jd = model.encode([JD_QUERY], normalize_embeddings=True).astype('float32')[0]
print(f'encoded {emb.shape[0]} x {emb.shape[1]} in {time.time()-t0:.1f}s')

os.makedirs('artifacts', exist_ok=True)
np.save('artifacts/candidate_embeddings.npy', emb)
np.save('artifacts/jd_embedding.npy', jd)
with open('artifacts/candidate_ids.txt', 'w') as f:
    f.write('\n'.join(ids))
with open('artifacts/model_name.txt', 'w') as f:
    f.write('BAAI/bge-small-en-v1.5')
print('saved:', os.listdir('artifacts'))
# sanity: cosine of JD vs first few candidates
print('sample JD cosines:', (emb[:5] @ jd).round(3))

In [ ]:
import shutil
shutil.make_archive('redrob_artifacts', 'zip', 'artifacts')
from google.colab import files
files.download('redrob_artifacts.zip')
print('Download started: redrob_artifacts.zip')